This script reads relevant datasets from the PROMICE/day folder. Only datasets from stations that are located on the icesheet (filtered the PROMICE aws_sites_metadata table for location == ice sheet). The dataset does NOT include data from local glaciers or tundra. 

Calculated mean position (lat, long) throughout the year per station and adds a column for (manually calculated) radiative temperature. 

Concatonates all resulting tables into one dataet (daily_prepped.csv) and another table of unique combinations of aws and year with their postition (ROMICE_yearly_avg_positions.csv)

In [ ]:
# In the acknowledgements: “Data from the Programme for Monitoring of the Greenland Ice Sheet (PROMICE) are provided by the Geological Survey of Denmark and Greenland (GEUS) at http://www.promice.dk. ZAC, LYN, FRE and NUK_K stations are financially supported by the Glaciobasis programme as part of the Greenland Ecosystem Monitoring project. The NUK_K station is serviced by Asiaq Greenland Survey. The WEG stations are paid for and maintained by Jakob Abermann at the Department of Geography and Regional Science of the University of Graz. The RED_Lv3 station is f inanced and maintained by Rainer Prinz at the Department of Atmospheric and Cryospheric Sciences of the University of Innsbruck. The SER_B station is paid for and serviced by Anders Bjørk at the Department of Geosciences and Natural Resource Management of the University of Copenhagen.”
# A reference to the dataset paper: Fausto, R. S., et al. 2021: “Programme for Monitoring of the Greenland Ice Sheet (PROMICE) automatic weather station data”, Earth Syst. Sci. Data, 13, 3819–3845, https://doi.org/10.5194/essd-13-3819-2021.
# doi = "How, P.; Lund, M.C.; Ahlstrøm, A.P.; Andersen, S.B.; Box, J.E.; Citterio, M.; Colgan, W.T.; Fausto, R.S.; Karlsson, N.B.; Jakobsen, J.; Jakobsgaard, H.T.; Larsen, S.H.; Mankoff, K.D.; Nielsen, R.B.; Rutishauser, A.; Shield, C.L.; Solgaard, A.M.; Stevens, I.T.; van As, D.; Vandecrux, B.; Abermann, J.; Bjørk, A.A.; Langley, K.; Lea, J.; Messerli, A.; Prinz, R., 2022, "PROMICE and GC-Net automated weather station data in Greenland", https://doi.org/10.22008/FK2/IW73UU, GEUS Dataverse, V28"

In [ ]:
import pandas as pd

station_names_str = ("CEN, CP1, DY2, EGP, HUM, JAR, KAN_L, KAN_M, KAN_T, KAN_U, KPC_L, KPC_U, NAE, NAU, NEM, NSE, NUK_L, NUK_N, NUK_U, QAS_A, QAS_L, QAS_M, QAS_U, RED_L, SCO_L, SCO_U, SDL, SDM, SWC, TAS_A, TAS_L, TAS_U, THU_L, THU_L2, THU_U, TUN, UPE_L, UPE_U, WEG_L")
station_names = station_names_str.split(", ")
file_names = []
table_list = []

path = "./Data/PROMICE/day/"


def create_filename(station_names):
    for name in station_names:
        example = "XXX_day.csv"
        filename = example.replace('XXX', name)
        file_names.append(filename)

create_filename(station_names)

def data_prep(df):
    # Convert time column to datetime format
    df['date'] = pd.to_datetime(df['time']).dt.date

    # Select relevant columns
    df = df[["date", "t_surf", "ulr", "dlr", "lat", "lon", "aws"]]

    # Drop rows with missing values in 't_surf
    df = df.dropna(subset=['t_surf'])

    # Group by year and calculate yearly mean of lat and lon
    df['year'] = pd.to_datetime(df['date']).dt.year
    df_means = df.groupby(['year', 'aws'], as_index=False).agg({
        'lat': 'mean',
        'lon': 'mean'
    })
    df = df.merge(df_means, on=['year', 'aws'], how='left', suffixes=('', '_yearly'))

    # Calculate radiative temperature from ulr
    e = 0.99 # Emissivity 
    df['rad_temp'] = ((df['ulr']-(1-e)*df['dlr']) / (5.67e-8 * e))**0.25-273.15
    # Fausto, R. S., van As, D., Mankoff, K. D., Vandecrux, B., Citterio, M., Ahlstrøm, A. P., Andersen, S. B., Colgan, W., Karlsson, N. B., Kjeldsen, K. K., Korsgaard, N. J., Larsen, S. H., Nielsen, S., Pedersen, A. {{\O{}}}., Shields, C. L., Solgaard, A. M., & Box, J. E. (2021). Programme for Monitoring of the Greenland Ice Sheet (PROMICE) automatic weather station data. Earth System Science Data, 13(8), 3819–3845. https://doi.org/10.5194/essd-13-3819-2021

    return df


def import_tables(file_names):
    # Import 
    for file_name in file_names: 
        table_name = file_name.replace("_day.csv", "")
        df = pd.read_csv(path+file_name)
        df['aws'] = table_name

    # Data preparation
        df = data_prep(df)

    # Store dataframe in global variables
        globals()[table_name]=df
        table_list.append(df)


import_tables(file_names)



In [4]:
# Dataframe that contains prepped data from all stations (incl. radiative temperature and averaged coordinates)

all_data = pd.concat(table_list, ignore_index=True)
# all_data.to_csv("./Data/PROMICE/output/daily_prepped.csv", index=False)

# Check if all data is contained
print(len(all_data))

def check_len(dfs):
    total_length = 0
    for df in dfs:
        total_length += len(df)
    return total_length

check_len(table_list)

121566


121566

In [5]:
positions = {"aws": [], "year": [], "lat": [], "lon": []}

for table in table_list: 
    grouped = table.groupby('year').first().reset_index()
    positions["aws"].extend(grouped['aws'].tolist())
    positions["year"].extend(grouped['year'].tolist())
    positions["lat"].extend(grouped['lat_yearly'].tolist())
    positions["lon"].extend(grouped['lon_yearly'].tolist())

positions_df = pd.DataFrame(positions)

# positions_df.to_csv("./Data/PROMICE/output/PROMICE_yearly_avg_positions.csv", index=False)

### TEST MAP. SKIP NEXT CELL IF UNNECESSARY

In [15]:
# TEST MAP - NOT TENTATIVE

import ee
import geemap
geemap.ee_initialize()

greenland = ee.Geometry.Polygon(
[[[-36.29516924635421, 83.70737243835941],
[-51.85180987135421, 82.75597137647488],
[-61.43188799635421, 81.99879137488564],
[-74.08813799635422, 78.10103528196419],
[-70.13305987135422, 75.65372336709613],
[-61.08032549635421, 75.71891096312955],
[-52.20337237135421, 60.9795530382023],
[-43.41430987135421, 58.59235996703347],
[-38.49243487135421, 64.70478286561182],
[-19.771731746354217, 69.72271161037442],
[-15.728762996354217, 76.0828635948066],
[-15.904544246354217, 79.45091003031243],
[-10.015872371354217, 81.62328742628017],
[-26.627200496354217, 83.43179828852398],
[-31.636966121354217, 83.7553561747887]]])

file = './Data/PROMICE/output/PROMICE_yearly_avg_positions.csv'

# Create FeatureCollection of AWS points from csv file containing station coordinates per year
def create_aws_points(file):
    df = pd.read_csv(file)
    features = []
    for index, row in df.iterrows():
        point = ee.Geometry.Point([row['lon'], row['lat']])
        feature = ee.Feature(point, {'id': row['aws'], 'year': row['year']})
        features.append(feature)
    return ee.FeatureCollection(features)

features = create_aws_points(file)


# Date
date = '2023-01-01'

img = ee.Image("NASA/ASTER_GED/AG100_003") 
landsat = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterDate('2023-01-01', '2023-01-31').first()
modis = ee.ImageCollection('MODIS/061/MOD11A1').filterDate('2023-01-01', '2023-01-31').first()
viirs = ee.ImageCollection("NASA/VIIRS/002/VNP21A1N").filterDate('2023-01-01', '2023-01-31').first()

# Apply correct scale and offset to emissivity band of modis

emis_31_modis = modis.select('Emis_31').multiply(0.002).add(0.49)
emis_16_viirs = viirs.select('Emis_16').multiply(0.002).add(0.49)

map = geemap.Map()

# map.addLayer(img, {}, "ASTER Image")
map.addLayer(landsat, {'band': ['ST_EMIS']}, "Landsat")
map.addLayer(emis_31_modis, {}, "MODIS Emissivity")
map.addLayer(emis_16_viirs, {}, "VIIRS Emissivity")
map.addLayer(features, {'color': 'red'}, "PROMICE AWS Locations")
map.centerObject(features, 5)

map


Map(center=[70.14497764119179, -45.83558556798536], controls=(WidgetControl(options=['position', 'transparent_…

KAN: 

In [ ]:
# df = CEN
# # df = df[["time", "lat", "lon", "alt"]]
# df['time'] = pd.to_datetime(df['time'], format='%Y-%m-%d')
# df['aws'] = 

